In [1]:
import pandas as pd
import numpy as np
from configs.settings import project_dir

In [2]:
p_dir = project_dir()
cleaned_dir_path = p_dir.CLEANED_DIR
featured_dir_path = p_dir.FEATURE_ENGINEERED

In [3]:
featured_dir_path

PosixPath('/home/arson/birdy/amit/noCartInsights/data/feature_engineered')

In [4]:
order_items_f = featured_dir_path/'featured_order_items.csv'
orders_f = featured_dir_path/'featured_orders.csv'
products_f = featured_dir_path/'featured_products.csv'

In [5]:
reviews = cleaned_dir_path/'cleaned_order_reviews.csv'
reviews_df = pd.read_csv(reviews,index_col=0)
reviews_df[['order_id','review_id']].duplicated().sum()

np.int64(0)

In [6]:
order_items_f_df = pd.read_csv(order_items_f,index_col=0)
order_items_f_df.sample(3)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,total_amount,seller_revenue,seller_order_count
95411,d86305c284cecff56c7ee9d3fbd4f8d6,1,af74cc53dcffc8384b29e7abfa41902b,213b25e6f54661939f11710a6fddb871,2018-04-19 23:10:02,79.8,13.92,93.72,18839.96,194
63473,90ee8b2a22737bb4eec65899b343eedc,1,e0d472a0e85ecfc7d8297c8bac6eb403,95ec4458365c4d11f452ccf538377619,2018-04-30 12:31:58,45.9,9.26,55.16,2303.43,31
50219,721fa08ba1208bce0d5642af802d2586,1,5127baa26f9d08000b80cc8b063fcd89,8bb48dc19fccaa8613b6229bf7f452a2,2018-05-07 15:53:46,16.4,19.32,35.72,6048.12,95


## T-Test hypothesis

### Do delayed orders receive significantly lower review scores compared to orders delivered on time?

##### To perform this hypothesis, we need order_id, is_delayed and reviews_score of an order. but we is_delayed column in orders_df and reviews_score in reviews dataframe, however, we have multiple reviews for the same orders in our reviews dataframe, so we are gonna take the mean the score for those case

In [7]:
reviews_df.info()

<class 'pandas.DataFrame'>
Index: 98410 entries, 0 to 99223
Data columns (total 8 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   review_id                98410 non-null  str  
 1   order_id                 98410 non-null  str  
 2   review_score             98410 non-null  int64
 3   review_comment_title     98410 non-null  str  
 4   review_comment_message   98410 non-null  str  
 5   review_creation_date     98410 non-null  str  
 6   review_answer_timestamp  98410 non-null  str  
 7   has_comment              98410 non-null  bool 
dtypes: bool(1), int64(1), str(6)
memory usage: 6.1 MB


In [8]:
reviews_df[reviews_df['order_id'].duplicated()]

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,has_comment
1119,46abf3ea0b2710ad41390fdb79c32d84,5040757d4e06a4be96d3827b860b4e7c,5,No Comment,No Comment,2017-11-07 00:00:00,2017-11-10 20:07:48,True
8108,40294ea5a778dc62080d6b3f55d361ce,e1bc1083cd7acd30d0576335373b907d,5,No Comment,No Comment,2018-03-23 00:00:00,2018-03-24 00:23:06,True
11807,2af839ae66a65959d6ae07775d0e7a35,bd859dd7a6a1b8c1df9fcb75c3604eaf,3,No Comment,bom,2018-03-28 00:00:00,2018-04-11 20:20:55,True
12594,bc85f39adbafaddfda29d372f3825873,f63a31c3349b87273468ff7e66852056,5,No Comment,No Comment,2018-01-11 00:00:00,2018-01-12 02:05:02,True
15921,d5bf42808fd0df2b766d5c8ece19b3b6,8ff88873f03f1912c00741f8e5ae6c79,5,No Comment,No Comment,2018-02-01 00:00:00,2018-02-05 18:51:59,True
...,...,...,...,...,...,...,...,...
98612,d23bba9a2f1d16e5505a02e5968c1e68,19fe6cd13dca5943f17abd2c37c46abd,5,No Comment,No Comment,2017-09-15 00:00:00,2017-09-22 16:39:24,True
98654,e28cc2a1bf48c11dbcc990894356bd82,1de86d094f7dd41cca13d246d3b7fd07,5,No Comment,No Comment,2017-11-15 00:00:00,2017-11-17 05:12:37,True
98677,8ae90d960cb871f44ebf423568e2985d,baed56f3eda9223b74c6cf175f05678e,5,No Comment,No Comment,2018-04-10 00:00:00,2018-04-10 19:10:35,True
98768,9c6dc4a9e9d3532bf73335c908a8b9be,f2f99bdf2e5cc73abc5e135a2ab1767e,5,No Comment,EU RECOMENDO SIM!\r\n,2018-04-03 00:00:00,2018-04-10 14:38:27,True


In [9]:
reviews_df['has_comment'].value_counts()

has_comment
True    98410
Name: count, dtype: int64

In [10]:
reviews_mean_df = reviews_df.groupby('order_id').agg(
    reviews_mean = ('review_score','mean')
)


In [11]:
reviews_df.shape

(98410, 8)

In [12]:
orders_f_df = pd.read_csv(orders_f)
orders_f_df.sample(3)

,order_id,customer_id,order_status,order_purchase,approved_at,delivered_carrier_date,delivered_customer_date,estimated_delivery_date,delivery_days,delivery_delay_days,is_delayed,customer_unique_id,customer_order_count,is_repeat_customer,order_value,order_count,total_spending_by_cust,avg_order_value_count
45568,e5bcc19af03dc9c714d381752e9f724f,f96d464a252356cef8521a343c9ef266,delivered,2017-07-16 11:48:22,2017-07-17 11:55:20,2017-07-18 12:34:50,2017-08-28 14:18:03,2017-08-16,42.0,12.0,1.0,306c929fdf6528e70f50643d82b850ac,1,0,355.08,1.0,355.08,355.080
5091,e470c6e4248f6c0e6b0c0567de82123f,e7aff9e12a7ac16af1b0c2dd61de3578,delivered,2018-04-23 10:59:47,2018-04-24 18:35:56,2018-04-25 00:04:00,2018-05-02 16:59:20,2018-05-16,7.0,-14.0,0.0,16e6739f107fe11620beee49d20c2bd6,1,0,37.69,1.0,37.69,37.690
6791,275322de6426ec5071b219b77fba6a65,ad970b9bf9eb36c6faa345affe1cf138,delivered,2018-01-16 23:21:40,2018-01-16 23:29:47,2018-01-17 21:54:38,2018-01-25 11:58:52,2018-02-15,8.0,-21.0,0.0,25cac425f64ae1b78c60c2afae3ec384,2,1,48.95,1.0,140.81,70.405


In [13]:
t_test_df = reviews_mean_df.merge(
    orders_f_df[['order_id','is_delayed']],
    on='order_id',
    how='left'
)

In [14]:
t_test_df.sample(5)

,order_id,reviews_mean,is_delayed
96213,fb01e8d8215d266abffb5ca379063244,3.0,0.0
42823,6fc7f8b453f5c366762afbf392ef9b37,5.0,0.0
39703,67aad3026d50e2928d6925136a0be9e7,5.0,0.0
50383,83d9d9a85946d3089b6b360120a54dcf,1.0,0.0
37,001862358bf858722e1e2ae000cfed8b,5.0,0.0


In [15]:
orders_f_df['is_delayed'].value_counts()

is_delayed
0.0    88652
1.0     7826
Name: count, dtype: int64

In [16]:
orders_f_df.loc[
    orders_f_df['order_id']=='0e6b2d0fe443a6d38c0f6447f4eb2262'
]

,order_id,customer_id,order_status,order_purchase,approved_at,delivered_carrier_date,delivered_customer_date,estimated_delivery_date,delivery_days,delivery_delay_days,is_delayed,customer_unique_id,customer_order_count,is_repeat_customer,order_value,order_count,total_spending_by_cust,avg_order_value_count
55515,0e6b2d0fe443a6d38c0f6447f4eb2262,35bfa7ce75679810efa7d433811c6945,unavailable,2017-11-03 20:44:04,2017-11-07 08:30:54,NaN,NaN,2017-11-28,NaN,NaN,NaN,bb0aea12660bf8bd536b39fddbc336ee,2,1,NaN,NaN,95.67,95.67


In [17]:
t_test_df['is_delayed'].isnull().sum()

np.int64(2791)

#### for our hypothesis, we need only those orders that are delivered to customers, not the NaN, these are orders that were never delivered to the customers, 
---> there are 2791 rows or entries that were never delivered to the customers, so we have to drop those rows. 

In [18]:
t_test_df['is_delayed'].isnull().sum()

np.int64(2791)

In [19]:
t_test_df.shape

(98128, 3)

In [20]:
t_test_df = t_test_df.dropna(subset=['is_delayed'])

In [21]:
t_test_df.shape

(95337, 3)

In [22]:
# avg review score for delayed orders and on-time orderrs
t_test_score = t_test_df.groupby('is_delayed').agg(
    mean_score = ('reviews_mean','mean')
)

In [23]:
t_test_df.sample()

,order_id,reviews_mean,is_delayed
84138,db52cc46b34b6c79d06569a1f7163d7b,5.0,0.0


In [24]:
t_test_score = t_test_df.groupby('is_delayed').agg(
    review_mean = (
        'reviews_mean','mean'
    ),
    review_std = (
        'reviews_mean','std'
    ),
    order_count = (
        'reviews_mean','count'
    )
)
t_test_score

,review_mean,review_std,order_count
is_delayed,,,
0.0,4.296253,1.144490,87714
1.0,2.567559,1.657945,7623


#### from t_test_score table, we can clearly see that for delayed orders, review score mean is appox is 2.56 and for on-time orders, mean_review-score is 4.3

H0 --> There is no difference between in review scores between delayed orders and on-time orders.

H1 --> Delayed orders receive low reviews scores as compared to scores of on-time orders. 

In [27]:
t_test_score

,review_mean,review_std,order_count
is_delayed,,,
0.0,4.296253,1.144490,87714
1.0,2.567559,1.657945,7623


In [28]:
t_test_df

,order_id,reviews_mean,is_delayed
0,00010242fe8c5a6d1ba2dd792cb16214,5.0,0.0
1,00018f77f2f0320c557190d7a144bdd3,4.0,0.0
2,000229ec398224ef6ca0657da4fc703e,5.0,0.0
3,00024acbcdf0a6daa1e931b038114c75,4.0,0.0
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.0,0.0
...,...,...,...
98123,fffc94f6ce00a00581880bf54a75a037,5.0,0.0
98124,fffcd46ef2263f404302a634eb57f7eb,5.0,0.0
98125,fffce4705a9662cd70adb13d4a31832d,5.0,0.0
98126,fffe18544ffabc95dfada21779c9644f,5.0,0.0


In [29]:
on_time = t_test_df[t_test_df['is_delayed'] == 0]['reviews_mean']
delayed = t_test_df[t_test_df['is_delayed'] == 1]['reviews_mean']

In [30]:
from scipy.stats import ttest_ind

t_stats, p_value = ttest_ind(
    on_time,
    delayed,
    equal_var=False
)

In [31]:
print("t-statistic:", t_stats)
print("p-value:", p_value)

t-statistic: 89.20710064839675
p-value: 0.0
